# Fine-Tune Your Own Llama Model 3.2 1B in a Colab Notebook

A practical introduction to LLM fine-tuning

![](https://archive.is/0iIXL/f587d66c7324054f5ae1e81d7a5736567e8c15c8.webp)

# Background on fine-tuning LLMs

![](https://archive.is/0iIXL/5f30742c57ad532b4cda9f1b48790dbcc7d00a85.webp)

**Summary:**

1. **LLM Pretraining:**
   - Large Language Models (LLMs) are pretrained on extensive text corpora.
   - Llama 2 was pretrained on a dataset of .....
   - Pretraining is resource-intensive and time-consuming (...)

2. **Auto-Regressive Prediction:**
   - Llama 2 and 3 , an auto-regressive model, predicts the next token in a sequence.
   - Auto-regressive models lack usefulness in providing instructions, leading to the need for instruction tuning.

3. **Fine-Tuning Techniques:**
   - Instruction tuning uses two main fine-tuning techniques:
     a. Supervised Fine-Tuning (SFT): Trained on instruction-response datasets, minimizing differences between generated and actual responses.
     b. Reinforcement Learning from Human Feedback (RLHF): Trained to maximize rewards based on human evaluations.

4. **RLHF vs. SFT:**
   - RLHF captures complex human preferences but requires careful reward system design and consistent human feedback.
   - Direct Preference Optimization (DPO) might be a future alternative to RLHF.
   - SFT can be highly effective when the model hasn't encountered specific data during pretraining.

5. **Effective SFT Example:**
   - LIMA paper showed improved performance of LLaMA v1 model over GPT-3 by fine-tuning on a small high-quality dataset.
   - Data quality and model size (e.g., 65b parameters) are crucial for successful fine-tuning.

6. **Importance of Prompt Templates:**
   - Prompt templates structure inputs: system prompt, user prompt, additional inputs, and model answer.
   - Llama 2's template example: <s>[INST] <<SYS>> System prompt <</SYS>> User prompt [/INST] Model answer </s>
   - Different templates (e.g., Alpaca, Vicuna,ChatGPT-style) have varying impacts.

7. **Reformatting for Llama 3:**
   - Converting instruction dataset to Llama 2's template is important.
   - The tutorial author already reformatted a dataset for this purpose.

8. **Base Llama 2 Model vs. Chat Version:**
   - Specific prompt templates not necessary for base Llama 2 model, unlike the chat version.

(Note: LLMs = Large Language Models, SFT = Supervised Fine-Tuning, RLHF = Reinforcement Learning from Human Feedback, DPO = Direct Preference Optimization)

### Fine-Tuning Llama 2 (1B Parameters) with VRAM Limitations using QLoRA

In this section, the objective is to fine-tune the **Llama 2 1B** model using a **T4 GPU (16 GB VRAM)**.  
Although the model is relatively small and can fit into memory with standard fine-tuning, **parameter-efficient fine-tuning (PEFT)** methods such as **LoRA** and **QLoRA** remain highly advantageous. These techniques **reduce VRAM usage**, **speed up training**, and **improve stability** on limited hardware.  

The chosen approach, **QLoRA (Quantized Low-Rank Adaptation)**, performs fine-tuning in **4-bit precision**, enabling efficient training while maintaining performance comparable to full fine-tuning in FP16 precision.

---

### Implementation Overview

1. **Environment Setup:**  
   The fine-tuning process leverages the **Hugging Face ecosystem**, including the following libraries:  
   `transformers`, `accelerate`, `peft`, `trl`, and `bitsandbytes`.

2. **Installation and Library Loading:**  
   The required dependencies are installed and imported following **Younes Belkada’s GitHub Gist**, which provides a lightweight setup for QLoRA-based fine-tuning.

3. **Fine-Tuning Objective:**  
   The process involves applying low-rank adapters to specific attention layers of the Llama 2 model while keeping base weights frozen.  
   This enables efficient **instruction-tuning** or **domain adaptation** on modest hardware.

---

> **Note:**  
> The **Llama 2 1B** model typically consumes around **4–6 GB of VRAM** in FP16 precision, and significantly less when quantized to 4-bit mode.  
> Using **QLoRA**, only a small number of adapter parameters are updated, making this method ideal for fine-tuning large language models on GPUs with limited memory.


In [1]:
!pip install -q accelerate==0.21.0 peft==0.4.0 bitsandbytes transformers==4.31.0 trl==0.4.7
!pip install -U bitsandbytes
#!pip install datasets bitsandbytes trl
!pip install transformers==4.55.2 peft==0.17.0 accelerate==1.10.0 trl==0.21.0 bitsandbytes==0.47.0 datasets==4.0.0 huggingface-hub==0.34.4 safetensors==0.6.2 pandas==2.2.2 matplotlib==3.10.0 numpy==2.0.2


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (tokenizers)
  Using cached bitsandbytes-0.48.1-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
Using cached bitsandbytes-0.48.1-py3-none-manylinux_2_24_x86_64.whl (60.1 MB)
  Attempting uninstall: bitsandbytes
    Found existing installation: bitsandbytes 0.47.0
    Uninstalling bitsandbytes-0.47.0:
      Successfully uninstalled bitsandbytes-0.47.0
  Using cached bitsandbytes-0.47.0-py3-none-manylinux_2_24_x86_64.whl.metadata (11 kB)
Using cached bitsandbytes-0.47.0-p

In [2]:
# Import necessary packages for the fine-tuning process
import os                          # Operating system functionalities
import torch                       # PyTorch library for deep learning
from datasets import load_dataset  # Loading datasets for training
from transformers import (
    AutoModelForCausalLM,          # AutoModel for language modeling tasks
    AutoTokenizer,                # AutoTokenizer for tokenization
    BitsAndBytesConfig,           # Configuration for BitsAndBytes
    HfArgumentParser,             # Argument parser for Hugging Face models
    TrainingArguments,            # Training arguments for model training
    pipeline,                     # Creating pipelines for model inference
    logging,                      # Logging information during training
)
from peft import LoraConfig, PeftModel  # Packages for parameter-efficient fine-tuning (PEFT)
from trl import SFTTrainer         # SFTTrainer for supervised fine-tuning

In [3]:
# !pip install -q datasets
!huggingface-cli login

⚠️  Warning: 'huggingface-cli login' is deprecated. Use 'hf auth login' instead.

    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    A token is already saved on your machine. Run `hf auth whoami` to get more information or `hf auth logout` if you want to log out.
    Setting a new token will erase the existing one.
    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add t

---
* **Section 1:** Parameters to tune
    * Load a meta-llama/Llama-3.2-1B-Instruct model and train it on the AgentPublic/piaf dataset.
    * The dataset contains 3,500 samples.
    * You can find more information about the dataset in hugginface .
    * Feel free to use a different dataset.
* **Section 2:** QLoRA parameters
    * QLoRA will use a rank of 64 with a scaling parameter of 16.
    * See this article for more information about LoRA parameters.
    * The Llama 3 model will be loaded directly in 4-bit precision using the NF4 type.
    * The model will be trained for one epoch.
* **Section 3:** Other parameters
    * To get more information about the other parameters, check the [TrainingArguments](https://archive.is/o/0iIXL/https://huggingface.co/docs/transformers/main_classes/trainer%23transformers.TrainingArguments), [PeftModel](https://archive.is/o/0iIXL/https://huggingface.co/docs/peft/package_reference/peft_model), and [SFTTrainer](https://archive.is/o/0iIXL/https://huggingface.co/docs/trl/main/en/sft_trainer) documentation.

In [4]:
# The model that you want to train from the Hugging Face hub
#model_name = "NousResearch/Llama-2-7b-hf"
#model_name = "meta-llama/Llama-3.2-1B"
model_name ="meta-llama/Llama-3.2-1B-Instruct"

# The instruction dataset to use
#dataset_name = "mlabonne/guanaco-llama2-1k"
dataset_name ="AgentPublic/piaf"

# Fine-tuned model name
# Change the name !!!!!
new_model = "llama-1B-lora64-piaf_test_1"

In [5]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",   # ou "cuda" si tu veux forcer GPU
    torch_dtype="auto"
)
prompt = "Qui est Jacques Chirrac ?"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(**inputs, max_new_tokens=200, temperature=0.7)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Qui est Jacques Chirrac? Jacques Chirac est un homme de politique français, né le 29 mars 1942 à Paris. Il est membre du Parti socialiste des députés (PS) et a occupé divers postes dans le gouvernement français.

Avant de devenir Premier ministre, Chirac a occupé divers postes importants dans le gouvernement, notamment :

* Ministre de la Défense nationale (1981-1984)
* Ministre des Affaires étrangères (1984-1986)
* Premier ministre (1995-2000)
* Président du Conseil des ministres (2000-2002)

En tant que Premier ministre, Chirac a mis en place plusieurs politiques importantes, notamment :

* La politique des deux tiers (1995-1997), qui visait à réduire la dette publique et à améliorer la qualité de la vie des Français
* La


In [6]:
################################################################################
# QLoRA parameters
################################################################################

# LoRA attention dimension
lora_r = 64

# Alpha parameter for LoRA scaling
lora_alpha = 16

# Dropout probability for LoRA layers
lora_dropout = 0.1

In [7]:
################################################################################
# bitsandbytes parameters
################################################################################

# Activate 4-bit precision base model loading
use_4bit = True

# Compute dtype for 4-bit base models
bnb_4bit_compute_dtype = "float16"

# Quantization type (fp4 or nf4)
bnb_4bit_quant_type = "nf4"

# Activate nested quantization for 4-bit base models (double quantization)
use_nested_quant = False

In [8]:
################################################################################
# TrainingArguments parameters
################################################################################

# Output directory where the model predictions and checkpoints will be stored
output_dir = "./results"

# Number of training epochs
num_train_epochs = 1

# Enable fp16/bf16 training (set bf16 to True with an A100)
fp16 = False
bf16 = False

# Batch size per GPU for training
per_device_train_batch_size = 2

# Batch size per GPU for evaluation
per_device_eval_batch_size = 2

# Number of update steps to accumulate the gradients for
gradient_accumulation_steps = 1

# Enable gradient checkpointing
gradient_checkpointing = True

# Maximum gradient normal (gradient clipping)
max_grad_norm = 0.3

# Initial learning rate (AdamW optimizer)
learning_rate = 2e-4

# Weight decay to apply to all layers except bias/LayerNorm weights
weight_decay = 0.001

# Optimizer to use
optim = "paged_adamw_32bit"

# Learning rate schedule (constant a bit better than cosine)
lr_scheduler_type = "constant"

# Number of training steps (overrides num_train_epochs)
max_steps = -1

# Ratio of steps for a linear warmup (from 0 to learning rate)
warmup_ratio = 0.03

# Group sequences into batches with same length
# Saves memory and speeds up training considerably
group_by_length = True

# Save checkpoint every X updates steps
save_steps = 25

# Log every X updates steps
logging_steps = 25

In [9]:
################################################################################
# SFT parameters
################################################################################

# Maximum sequence length to use
max_seq_length = 64

# Pack multiple short examples in the same input sequence to increase efficiency
packing = False

# Load the entire model on the GPU 0
device_map = {"": 0}


1. **Loading the Dataset:**
   The first step involves loading the preprocessed dataset. This dataset will be used for fine-tuning. Preprocessing might involve reformatting prompts, filtering out low-quality text, and combining multiple datasets if needed.

2. **Configuring BitsAndBytes for 4-bit Quantization:**
   The `BitsAndBytesConfig` is set up to enable 4-bit quantization. This configuration is crucial for reducing the memory usage during fine-tuning.

3. **Loading Llama 2 Model and Tokenizer in 4-bit Precision:**
   The Llama 2 model is loaded with 4-bit precision, which significantly reduces the memory footprint. The corresponding tokenizer is also loaded to preprocess the text data.

4. **Loading Configurations and Initializing SFTTrainer:**
   - The configurations needed for QLoRA, which is a parameter-efficient fine-tuning technique, are loaded.
   - Regular training parameters are set up.
   - The `SFTTrainer` is initialized with all the loaded configurations and parameters. This trainer will manage the supervised fine-tuning process.

5. **Start of Training:**
   After all the necessary components are loaded and configured, the training process begins. The `SFTTrainer` takes care of fine-tuning the Llama 2 model using the specified dataset, configurations, and parameters.
   
  These steps collectively set up the environment for fine-tuning a Llama 2 model with 7 billion parameters in 4-bit precision using the QLoRA technique, thus optimizing for VRAM limitations while maintaining model performance.

In [10]:
# Step 1 : Load dataset (you can process it here)
dataset = load_dataset(dataset_name, split="train")

In [11]:
# Step 2 :Load tokenizer and model with QLoRA configuration
compute_dtype = getattr(torch, bnb_4bit_compute_dtype)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=use_4bit,
    bnb_4bit_quant_type=bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=use_nested_quant,
)

In [12]:
# Step 3 :Check GPU compatibility with bfloat16
if compute_dtype == torch.float16 and use_4bit:
    major, _ = torch.cuda.get_device_capability()
    if major >= 8:
        print("=" * 80)
        print("Your GPU supports bfloat16: accelerate training with bf16=True")
        print("=" * 80)

In [13]:
# Step 4 :Load base model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map=device_map
)
model.config.use_cache = False
model.config.pretraining_tp = 1

In [14]:
# Step 5 :Load LLaMA tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.add_special_tokens({'pad_token': '[PAD]'})
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [15]:
# Step 6 :Load LoRA configuration
peft_config = LoraConfig(
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    r=lora_r,
    bias="none",
    task_type="CAUSAL_LM",
)

In [16]:
# Step 7 :Set training parameters
training_arguments = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=2,  # Reduced batch size
    gradient_accumulation_steps=gradient_accumulation_steps,
    optim=optim,
    save_steps=save_steps,
    logging_steps=logging_steps,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    fp16=fp16,
    bf16=bf16,
    max_grad_norm=max_grad_norm,
    max_steps=max_steps,
    warmup_ratio=warmup_ratio,
    group_by_length=group_by_length,
    lr_scheduler_type=lr_scheduler_type,
    report_to="tensorboard"
)

## Dataset has to be adapted for the training

In [17]:
from datasets import load_dataset

# Inspect the dataset to see available columns
print(dataset.column_names)

# Charger dataset
dataset = load_dataset("etalab-ia/piaf", split="train[:3000]")  # petit échantillon pour test

# Fonction de prétraitement
def preprocess(example):
    context = example["context"]
    question = example["question"]
    answer = example["answers"]["text"][0] if example["answers"]["text"] else ""
    return {
        "text": f"Contexte : {context}\n\nQuestion : {question}\n\nRéponse : {answer}"
    }

dataset = dataset.map(preprocess)


['id', 'title', 'context', 'question', 'answers']


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/650k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3835 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

In [18]:
# Step 8 :Set supervised fine-tuning parameters
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    args=training_arguments,
)

Adding EOS to train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

In [19]:
import torch
torch.cuda.empty_cache()


In [ ]:
# Step 9 :Train model
trainer.train()

# Step 10 :Save trained model
trainer.model.save_pretrained(new_model)

Step,Training Loss
25,2.501300
50,2.414200


In [ ]:
%load_ext tensorboard
%tensorboard --logdir results/runs

In [ ]:
model.push_to_hub(new_model, use_temp_dir=False)
tokenizer.push_to_hub(new_model, use_temp_dir=False)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from datasets import load_dataset

# --- 1️⃣ Charger le modèle fine-tuné ---
model_name = "meta-llama/Llama-3.2-1B-Instruct"
new_model = "llama-1B-lora64-piaf_test22102025"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(new_model).to("cuda")  # ou "cpu"

# --- 2️⃣ Charger un petit batch de questions pour tester ---
dataset_name = "AgentPublic/piaf"
dataset = load_dataset(dataset_name, split="train[:10]")  # 10 exemples pour test rapide

# --- 3️⃣ Fonction pour générer la réponse ---
def generate_answer(context, question, max_new_tokens=100):
    prompt = f"Contexte : {context}\n\nQuestion : {question}\n\nRéponse :"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.7,
        do_sample=True,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )
    # Supprimer le prompt pour ne garder que la réponse
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = generated_text[len(prompt):].strip()
    return answer

# --- 4️⃣ Tester le modèle sur le dataset PIAF ---
for ex in dataset:
    context = ex["context"]
    question = ex["question"]
    reference_answer = ex["answers"]["text"][0] if ex["answers"]["text"] else ""

    generated_answer = generate_answer(context, question)

    print("----")
    print(f"Q: {question}")
    print(f"Référence: {reference_answer}")
    print(f"Généré: {generated_answer}")


In [ ]:
# --- Initialisation des modèles ---
model_base = model  # modèle de base LLaMA-1B-Instruct
model_lora = AutoModelForCausalLM.from_pretrained(new_model).to("cuda")  # modèle fine-tuné LoRA
tokenizer_base = tokenizer  # même tokenizer pour les deux

print("=== Chat comparatif : Base vs Fine-tuned LoRA ===")
print("Tape 'quit' pour sortir.\n")

while True:
    user_input = input("👤 Toi : ")
    if user_input.lower() in {"quit", "exit"}:
        break

    prompt = f"Question : {user_input}\nRéponse :"

    # --- Réponse modèle de base ---
    inputs_base = tokenizer_base(prompt, return_tensors="pt").to(model_base.device)
    outputs_base = model_base.generate(
        **inputs_base,
        max_new_tokens=200,
        temperature=0.1, # 0.7
        pad_token_id=tokenizer_base.eos_token_id
    )
    response_base = tokenizer_base.decode(outputs_base[0], skip_special_tokens=True)
    response_base = response_base.split("Réponse :")[-1].strip()

    # --- Réponse modèle LoRA fine-tuné ---
    inputs_lora = tokenizer_base(prompt, return_tensors="pt").to(model_lora.device)
    outputs_lora = model_lora.generate(
        **inputs_lora,
        max_new_tokens=200,
        temperature=0.1, # 0.7
        pad_token_id=tokenizer_base.eos_token_id
    )
    response_lora = tokenizer_base.decode(outputs_lora[0], skip_special_tokens=True)
    response_lora = response_lora.split("Réponse :")[-1].strip()

    # --- Affichage côte à côte ---
    print(f"🤖 Base : {response_base}")
    print(f"🤖 Fine-tuned LoRA : {response_lora}\n")



In [ ]:
import torch

# Vider le cache GPU
torch.cuda.empty_cache()

# Vérifier la mémoire GPU utilisée (optionnel)
print(torch.cuda.memory_summary())

